In [1]:
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [2]:
# ---- Dummy data  (36 train + 36 eval samples) ----

def make_blob(h=40, w=20, cx=10, cy=20, r=5, noise=0.05, seed=0):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:h, 0:w]
    mask = np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * r**2))
    return np.clip(mask + rng.normal(0, noise, mask.shape), 0, 1).astype(np.float32)

def make_overhead(seed=0):
    rng = np.random.default_rng(seed)
    img = np.full((200, 100, 3), 210, dtype=np.uint8)
    img[::20, :] = 195; img[:, ::10] = 195
    cx, cy = rng.integers(15, 85), rng.integers(40, 160)
    for dy in range(-15, 15):
        for dx in range(-10, 10):
            if 0 <= cy+dy < 200 and 0 <= cx+dx < 100:
                img[cy+dy, cx+dx] = [60, 80, 200]
    return Image.fromarray(img)

rng_meta = np.random.default_rng(0)
SAMPLES = []
for i in range(72):
    split  = 'train' if i < 36 else 'eval'
    SAMPLES.append(dict(
        sample_idx = i,
        split      = split,
        x          = round(float(rng_meta.uniform(0.5, 9.5)), 1),
        y          = round(float(rng_meta.uniform(0.5, 9.5)), 1),
        object     = 'cube',
        n_objects  = int(rng_meta.integers(1, 3)),
    ))

EPOCHS = [10, 20, 30, 40, 50]

predictions = {'train': {}, 'eval': {}}
for ep in EPOCHS:
    quality = ep / max(EPOCHS)
    for split in ['train', 'eval']:
        grp = [s for s in SAMPLES if s['split'] == split]
        mask_true = np.array([make_blob(cx=int(s['x']*2), cy=int(s['y']*4),
                                        noise=0, seed=s['sample_idx']) for s in grp])
        mask_pred = np.array([make_blob(cx=int(s['x']*2), cy=int(s['y']*4),
                                        noise=max(0.25 - quality*0.22, 0.02),
                                        seed=s['sample_idx'] + ep) for s in grp])
        predictions[split][ep] = dict(
            mask_true   = mask_true,
            mask_pred   = mask_pred,
            sample_idx  = np.array([s['sample_idx'] for s in grp]),
            x_position  = np.array([s['x']          for s in grp]),
            y_position  = np.array([s['y']          for s in grp]),
            object_type = np.array([s['object']     for s in grp]),
            n_objects   = np.array([s['n_objects']  for s in grp]),
        )

overhead_images = {s['sample_idx']: make_overhead(s['sample_idx']) for s in SAMPLES}

metrics_by_epoch = {}
for ep in EPOCHS:
    q = ep / max(EPOCHS)
    metrics_by_epoch[ep] = {
        'metrics/train/mask/iou': round(0.30 + q * 0.52, 3),
        'metrics/eval/mask/iou':  round(0.22 + q * 0.42, 3),
        'metrics/train/mask/mse': round(0.42 - q * 0.32, 3),
        'metrics/eval/mask/mse':  round(0.48 - q * 0.32, 3),
        'loss/train/total':       round(0.85 - q * 0.55, 3),
    }

run_config = dict(loss='focal', gamma=2.0, decoder='mlp', d_model=64,
                  batch_size=8, lr=1e-4, seed=42, n_params=45312)
RUN_ID = 'viz-test-dummy'
print(f'Dummy data ready: {sum(s["split"]=="train" for s in SAMPLES)} train, {sum(s["split"]=="eval" for s in SAMPLES)} eval')

Dummy data ready: 36 train, 36 eval


In [9]:
# ---- Visualization helpers ----

TOTAL_ROWS = 3
TOTAL_COLS = 10
assert TOTAL_COLS % 2 == 0, "TOTAL_COLS should be even for side-by-side mode"

def cols_for(subset):
    return TOTAL_COLS//2 if subset == 'both' else TOTAL_COLS

def get_npz_row(split, epoch, sample_idx):
    epochs = sorted(predictions[split].keys())
    epoch  = min(epochs, key=lambda e: abs(e - epoch))
    npz    = predictions[split][epoch]
    idxs   = npz['sample_idx'].tolist()
    if sample_idx not in idxs: return None, None, None
    i = idxs.index(sample_idx)
    meta = dict(
        x_position = float(npz['x_position'][i]),
        y_position = float(npz['y_position'][i]),
        object     = str(npz['object_type'][i]),
        n_objects  = int(npz['n_objects'][i]),
    )
    return npz['mask_true'][i], npz['mask_pred'][i], meta

def calc_iou(mt, mp, t=0.5):
    inter = float(np.mean((mt > t) & (mp > t)))
    union = float(np.mean((mt > t) | (mp > t)))
    return inter / max(union, 1e-6)

def cell_title(split, sid, mt, mp, meta):
    tag  = "TR" if split == "train" else "EV"
    obj  = meta["object"]    if meta else ""
    nobj = meta["n_objects"] if meta else ""
    pos  = f'x={meta["x_position"]:.1f} y={meta["y_position"]:.1f}' if meta else ""
    return f'{tag}#{sid} {obj} n={nobj}<br>{pos}'

def add_cell(fig, row, col, split, sample_idx, epoch, mode, opacity):
    mt, mp, _ = get_npz_row(split, epoch=epoch, sample_idx=sample_idx)
    if mt is None: return
    oh = overhead_images.get(sample_idx)
    oh_arr = np.array(oh) if oh else np.full((200, 100, 3), 220, dtype=np.uint8)

    fig.add_trace(go.Image(z=oh_arr, hoverinfo='skip'), row=row, col=col)

    if mode == 'overlay':
        fig.add_trace(go.Heatmap(z=mp, colorscale='Hot', zmin=0, zmax=1, opacity=opacity,
            showscale=False, hovertemplate='pred: %{z:.3f}<extra></extra>'), row=row, col=col)
        fig.add_trace(go.Contour(z=mt, contours_coloring='lines',
            colorscale=[[0,'lime'],[1,'lime']], line_width=2, showscale=False,
            hovertemplate='gt: %{z:.3f}<extra></extra>'), row=row, col=col)

    elif mode == 'side-by-side':
        gt_half = np.concatenate([mt, np.full_like(mt,  np.nan)], axis=1)
        pr_half = np.concatenate([np.full_like(mp, np.nan), mp],  axis=1)
        fig.add_trace(go.Heatmap(z=gt_half, colorscale='gray', zmin=0, zmax=1,
            showscale=False, hovertemplate='gt: %{z:.3f}<extra></extra>'), row=row, col=col)
        fig.add_trace(go.Heatmap(z=pr_half, colorscale='Hot',  zmin=0, zmax=1,
            showscale=False, hovertemplate='pred: %{z:.3f}<extra></extra>'), row=row, col=col)

    else:
        fig.add_trace(go.Heatmap(z=np.abs(mt - mp), colorscale='RdBu_r', zmin=0, zmax=1,
            opacity=opacity, showscale=False,
            hovertemplate='|gt-pred|: %{z:.3f}<extra></extra>'), row=row, col=col)

def get_split_idxs(split, n):
    return [s['sample_idx'] for s in SAMPLES if s['split'] == split][:n]

def build_figure(epoch, mode, opacity, subset):
    m  = metrics_by_epoch.get(epoch, {})
    fv = lambda k: f'{m[k]:.3f}' if k in m else '\u2014'

    cfg = run_config
    cfg_line = (f'<b>{RUN_ID}</b>  loss={cfg["loss"]}  gamma={cfg["gamma"]}  '
                f'decoder={cfg["decoder"]}  d_model={cfg["d_model"]}  '
                f'lr={cfg["lr"]}  n_params={cfg["n_params"]:,}')
    metrics_line = (f'Epoch {epoch}  |  '
                    f'Train IoU: <b>{fv("metrics/train/mask/iou")}</b>  '
                    f'Eval IoU: <b>{fv("metrics/eval/mask/iou")}</b>  |  '
                    f'Train MSE: <b>{fv("metrics/train/mask/mse")}</b>  '
                    f'Eval MSE: <b>{fv("metrics/eval/mask/mse")}</b>  |  '
                    f'Loss: <b>{fv("loss/train/total")}</b>')
    title_text = f'{cfg_line}<br><span style="font-size:12px;color:#444">{metrics_line}</span>'

    splits   = ['train', 'eval'] if subset == 'both' else [subset]
    c_per_sp = cols_for(subset)
    n_per_sp = TOTAL_ROWS * c_per_sp

    idxs_by_split = {sp: get_split_idxs(sp, n_per_sp) for sp in splits}

    subplot_titles = []
    for r in range(TOTAL_ROWS):
        for sp in splits:
            for c in range(c_per_sp):
                i   = r * c_per_sp + c
                idx = idxs_by_split[sp]
                if i < len(idx):
                    sid = idx[i]
                    mt, mp, meta = get_npz_row(sp, epoch, sid)
                    subplot_titles.append(cell_title(sp, sid, mt, mp, meta))
                else:
                    subplot_titles.append('')

    fig = make_subplots(
        rows=TOTAL_ROWS, cols=TOTAL_COLS,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.01,
        vertical_spacing=0.09,
    )

    for si, sp in enumerate(splits):
        col_offset = si * c_per_sp
        for i, sid in enumerate(idxs_by_split[sp]):
            r = i // c_per_sp + 1
            c = i %  c_per_sp + col_offset + 1
            add_cell(fig, r, c, sp, sid, epoch, mode, opacity)

    if subset == 'both':
        fig.add_annotation(text='<b>TRAIN</b>', x=0.25, y=1.04,
            xref='paper', yref='paper', showarrow=False,
            font=dict(size=13, color='#1a56db'))
        fig.add_annotation(text='<b>EVAL</b>', x=0.75, y=1.04,
            xref='paper', yref='paper', showarrow=False,
            font=dict(size=13, color='#c0392b'))
        fig.add_shape(type='line', x0=0.5, y0=0, x1=0.5, y1=1,
            xref='paper', yref='paper',
            line=dict(color='#aaa', width=1.5, dash='dot'))

    fig.update_layout(
        title=dict(text=title_text, font=dict(size=13, color='black'), x=0.0, xanchor='left'),
        height=170 * TOTAL_ROWS + 155,
        margin=dict(t=150, b=10, l=10, r=10),
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(color='#333', family='monospace'),
        showlegend=False,
    )
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
    for ann in fig.layout.annotations:
        if ann.text not in ('<b>TRAIN</b>', '<b>EVAL</b>'):
            ann.font.size  = 12
            ann.font.color = '#444'
    return fig

print('Helpers ready.')

Helpers ready.


In [10]:
# ---- Interactive widget ----

eval_epochs = sorted(predictions['eval'].keys())
step = eval_epochs[1] - eval_epochs[0] if len(eval_epochs) > 1 else 1

epoch_slider = widgets.IntSlider(
    value=eval_epochs[-1], min=eval_epochs[0], max=eval_epochs[-1], step=step,
    description='Epoch', continuous_update=False, layout=widgets.Layout(width='50%'))
play = widgets.Play(
    value=eval_epochs[0], min=eval_epochs[0], max=eval_epochs[-1], step=step,
    interval=700, description='\u25b6')
widgets.jslink((play, 'value'), (epoch_slider, 'value'))

subset_toggle = widgets.ToggleButtons(
    options=['both', 'train', 'eval'], value='both',
    description='Data', style={'button_width': '70px'})

mode_toggle = widgets.ToggleButtons(
    options=['overlay', 'side-by-side', 'difference'], value='overlay',
    description='Mode', style={'button_width': '105px'})

opacity_slider = widgets.FloatSlider(
    value=0.55, min=0.1, max=1.0, step=0.05,
    description='Opacity', continuous_update=False,
    layout=widgets.Layout(width='40%'))

fig_out = widgets.Output()

def show_fig(epoch, mode, opacity, subset):
    fig  = build_figure(epoch, mode, opacity, subset)
    html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    with fig_out:
        clear_output(wait=True)
        display(HTML(html))

def refresh(*_):
    epoch = min(eval_epochs, key=lambda e: abs(e - epoch_slider.value))
    show_fig(epoch, mode_toggle.value, opacity_slider.value, subset_toggle.value)

for w in [epoch_slider, mode_toggle, opacity_slider, subset_toggle]:
    w.observe(refresh, names='value')

show_fig(eval_epochs[-1], 'overlay', 0.55, 'both')

controls = widgets.VBox([
    widgets.HBox([play, epoch_slider]),
    widgets.HBox([subset_toggle, mode_toggle, opacity_slider]),
])
display(widgets.VBox([fig_out, controls]))